In [ ]:
pip install pandas gensim openpyxl regex ftfy scikit-learn


shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/CENG442_Assignment1


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-2263105412.py", line 1, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content/drive/MyDrive/CENG442_Assignment1')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another excep

In [ ]:
# -*- coding: utf-8 -*-
import re, html, unicodedata, pandas as pd
from pathlib import Path
try:
    from ftfy import fix_text
except Exception:
    def fix_text(s): return s

# --- Azerbaijani-aware lowercase
def lower_az(s: str) -> str:
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFC", s)
    s = s.replace("I", "ı").replace("İ", "i")
    s = s.lower().replace("i ̇","i")
    return s

# --- Regex patterns
HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_RE   = re.compile(r"(https?://\S+|www\.\S+)", re.I)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", re.I)
PHONE_RE = re.compile(r"\+?\d[\d\-\s\(\)]{6,}\d")
USER_RE  = re.compile(r"@\w+")
MULTI_PUNCT = re.compile(r"([!?.,;:])\1{1,}")
MULTI_SPACE = re.compile(r"\s+")
REPEAT_CHARS= re.compile(r"(.)\1{2,}", re.UNICODE)
TOKEN_RE = re.compile(
    r"[A-Za-zƏəĞğIıİiÖöÜüÇçŞşXxQq]+(?:'[A-Za-zƏəĞğIıİiÖöÜüÇçŞşXxQq]+)?"
    r"|<NUM>|URL|EMAIL|PHONE|USER|EMO_(?:POS|NEG)"
)

# --- Mini resources
EMO_MAP = {"🙂":"EMO_POS","😀":"EMO_POS","😍":"EMO_POS","😊":"EMO_POS","👍":"EMO_POS",
           "☹":"EMO_NEG","🙁":"EMO_NEG","😠":"EMO_NEG","😡":"EMO_NEG","👎":"EMO_NEG"}
SLANG_MAP = {"slm":"salam","tmm":"tamam","sagol":"sağol","cox":"çox","yaxsi":"yaxşı"}
NEGATORS  = {"yox","deyil","heç","qətiyyən","yoxdur"}

# --- Domain helpers (from Section 6)
NEWS_HINTS   = re.compile(r"\b(apa|trend|azertac|reuters|bloomberg|dha|aa)\b", re.I)
SOCIAL_HINTS = re.compile(r"\b(rt)\b|@|#|(?:😂|😍|😊|👍|👎|😡|🙂)")
REV_HINTS    = re.compile(r"\b(azn|manat|qiymət|aldım|ulduz|çox yaxşı|çox pis)\b", re.I)
PRICE_RE     = re.compile(r"\b\d+\s*(azn|manat)\b", re.I)
STARS_RE     = re.compile(r"\b([1-5])\s*ulduz\b", re.I)
POS_RATE     = re.compile(r"\bçox yaxşı\b")
NEG_RATE     = re.compile(r"\bçox pis\b")

def detect_domain(text: str) -> str:
    s = text.lower()
    if NEWS_HINTS.search(s): return "news"
    if SOCIAL_HINTS.search(s): return "social"
    if REV_HINTS.search(s):   return "reviews"
    return "general"

def domain_specific_normalize(cleaned: str, domain: str) -> str:
    if domain == "reviews":
        s = PRICE_RE.sub(" <PRICE> ", cleaned)
        s = STARS_RE.sub(lambda m: f" <STARS_{m.group(1)}> ", cleaned)
        s = POS_RATE.sub(" <RATING_POS> ", s)
        s = NEG_RATE.sub(" <RATING_NEG> ", s)
        return " ".join(s.split())
    return cleaned

def add_domain_tag(line: str, domain: str) -> str:
    return f"dom{domain} " + line

# --- Main cleaning
def normalize_text_az(s: str, numbers_to_token=True, keep_sentence_punct=False) -> str:
    if not isinstance(s, str): return ""
    for emo, tag in EMO_MAP.items(): s = s.replace(emo, f" {tag} ")
    s = fix_text(html.unescape(s))
    s = HTML_TAG_RE.sub(" ", s)
    s = URL_RE.sub(" URL ", s)
    s = EMAIL_RE.sub(" EMAIL ", s)
    s = PHONE_RE.sub(" PHONE ", s)
    s = re.sub(r"#([A-Za-z0-9_]+)", lambda m: " " + re.sub('([a-z])([A-Z])', r'\1 \2', m.group(1)) + " ", s)
    s = USER_RE.sub(" USER ", s)
    s = lower_az(s)
    s = MULTI_PUNCT.sub(r"\1", s)
    if numbers_to_token: s = re.sub(r"\d+", " <NUM> ", s)
    s = re.sub(r"[^\w\s<>'əğıöşüçƏĞIİÖŞÜÇxqXQ]" if not keep_sentence_punct else r"[^\w\s<>'əğıöşüçƏĞIİÖŞÜÇxqXQ.!?]", " ", s)
    s = MULTI_SPACE.sub(" ", s).strip()
    toks = TOKEN_RE.findall(s)
    norm, mark_neg = [], 0
    for t in toks:
        t = REPEAT_CHARS.sub(r"\1\1", t)
        t = SLANG_MAP.get(t, t)
        if t in NEGATORS:
            norm.append(t); mark_neg = 3; continue
        if mark_neg > 0 and t not in {"URL","EMAIL","PHONE","USER"}:
            norm.append(t + "_NEG"); mark_neg -= 1
        else: norm.append(t)
    norm = [t for t in norm if not (len(t)==1 and t not in {"o","e"})]
    return " ".join(norm).strip()

def map_sentiment_value(v, scheme: str):
    if scheme=="binary":
        try: return 1.0 if int(v)==1 else 0.0
        except: return None
    s=str(v).strip().lower()
    if s in {"pos","positive","1","müsbət","good","pozitiv"}: return 1.0
    if s in {"neu","neutral","2","neytral"}: return 0.5
    if s in {"neg","negative","0","mənfi","bad","neqativ"}: return 0.0
    return None

def process_file(in_path, text_col, label_col, scheme, out_two_col_path, remove_stopwords=False):
    df = pd.read_excel(in_path)
    for c in ["Unnamed: 0","index"]:
        if c in df.columns: df = df.drop(columns=[c])
    assert text_col in df.columns and label_col in df.columns, f"Missing columns in {in_path}"
    df = df.dropna(subset=[text_col])
    df = df[df[text_col].astype(str).str.strip().str.len()>0].drop_duplicates(subset=[text_col])
    df["cleaned_text"] = df[text_col].astype(str).apply(normalize_text_az)
    df["__domain__"] = df[text_col].astype(str).apply(detect_domain)
    df["cleaned_text"] = df.apply(lambda r: domain_specific_normalize(r["cleaned_text"], r["__domain__"]), axis=1)
    df["sentiment_value"] = df[label_col].apply(lambda v: map_sentiment_value(v, scheme))
    df = df.dropna(subset=["sentiment_value"])
    df["sentiment_value"] = df["sentiment_value"].astype(float)
    out_df = df[["cleaned_text","sentiment_value"]].reset_index(drop=True)
    Path(out_two_col_path).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_excel(out_two_col_path, index=False)
    print(f"Saved: {out_two_col_path} ({len(out_df)} rows)")

def build_corpus_txt(input_files, text_cols, out_txt="corpus_all.txt"):
    lines=[]
    for f, text_col in zip(input_files, text_cols):
        df=pd.read_excel(f)
        for raw in df[text_col].dropna().astype(str):
            dom=detect_domain(raw)
            s=normalize_text_az(raw, keep_sentence_punct=True)
            parts=re.split(r"[.!?]+", s)
            for p in parts:
                p=p.strip()
                if not p: continue
                p=re.sub(r"[^\w\səğıöşüçƏĞIİÖŞÜÇxqXQ]"," ",p)
                p=" ".join(p.split()).lower()
                if p: lines.append(f"dom{dom} "+p)
    with open(out_txt,"w",encoding="utf-8") as w:
        for ln in lines: w.write(ln+"\n")
    print(f"Wrote {out_txt} ({len(lines)} lines)")

if __name__=="__main__":
    CFG=[("labeled-sentiment.xlsx","text","sentiment","tri"),
         ("test__1_.xlsx","text","label","binary"),
         ("train__3_.xlsx","text","label","binary"),
         ("train-00000-of-00001.xlsx","text","labels","tri"),
         ("merged_dataset_CSV__1_.xlsx","text","labels","binary")]
    for fname,tcol,lcol,scheme in CFG:
        out=f"{Path(fname).stem}_2col.xlsx"
        process_file(fname,tcol,lcol,scheme,out)
    build_corpus_txt([c[0] for c in CFG],[c[1] for c in CFG],"corpus_all.txt")


OSError: [Errno 107] Transport endpoint is not connected: 'labeled-sentiment.xlsx'

In [ ]:
from gensim.models import Word2Vec, FastText
import pandas as pd
from pathlib import Path

files=[
 "labeled-sentiment_2col.xlsx",
 "test__1__2col.xlsx",
 "train__3__2col.xlsx",
 "train-00000-of-00001_2col.xlsx",
 "merged_dataset_CSV__1__2col.xlsx",
]
print("Çalıştı...")
sentences=[]
for f in files:
    df=pd.read_excel(f,usecols=["cleaned_text"])
    sentences.extend(df["cleaned_text"].astype(str).str.split().tolist())

Path("embeddings").mkdir(exist_ok=True)
w2v=Word2Vec(sentences=sentences,vector_size=300,window=5,min_count=3,sg=1,negative=10,epochs=10)
w2v.save("embeddings/word2vec.model")
ft=FastText(sentences=sentences,vector_size=300,window=5,min_count=3,sg=1,min_n=3,max_n=6,epochs=10)
ft.save("embeddings/fasttext.model")
print("Saved embeddings.")


Çalıştı...
Saved embeddings.


In [ ]:
import pandas as pd
from gensim.models import Word2Vec, FastText
from numpy import dot
from numpy.linalg import norm

w2v=Word2Vec.load("embeddings/word2vec.model")
ft =FastText.load("embeddings/fasttext.model")

seed_words=["yaxşı","pis","çox","bahalı","ucuz","mükəmməl","dəhşət","<PRICE>","<RATING_POS>"]
syn_pairs=[("yaxşı","əla"),("bahalı","qiymətli"),("ucuz","sərfəli")]
ant_pairs=[("yaxşı","pis"),("bahalı","ucuz")]

def lexical_coverage(model,tokens):
    vocab=model.wv.key_to_index
    return sum(1 for t in tokens if t in vocab)/max(1,len(tokens))

files=[
 "labeled-sentiment_2col.xlsx",
 "test__1__2col.xlsx",
 "train__3__2col.xlsx",
 "train-00000-of-00001_2col.xlsx",
 "merged_dataset_CSV__1__2col.xlsx",
]

def read_tokens(f):
    df=pd.read_excel(f,usecols=["cleaned_text"])
    return [t for row in df["cleaned_text"].astype(str) for t in row.split()]

print("== Lexical coverage ==")
for f in files:
    toks=read_tokens(f)
    print(f"{f}: W2V={lexical_coverage(w2v,toks):.3f}, FT={lexical_coverage(ft,toks):.3f}")

def pair_sim(model,pairs):
    vals=[]
    for a,b in pairs:
        try: vals.append(model.wv.similarity(a,b))
        except KeyError: pass
    return sum(vals)/len(vals) if vals else float('nan')

print("\n== Similarities ==")
print(f"Synonyms: W2V={pair_sim(w2v,syn_pairs):.3f}, FT={pair_sim(ft,syn_pairs):.3f}")
print(f"Antonyms: W2V={pair_sim(w2v,ant_pairs):.3f}, FT={pair_sim(ft,ant_pairs):.3f}")

def neighbors(model,word,k=5):
    try: return [w for w,_ in model.wv.most_similar(word,topn=k)]
    except KeyError: return []

print("\n== Nearest neighbors ==")
for w in seed_words:
    print(f"  W2V NN for '{w}':",neighbors(w2v,w))
    print(f"  FT  NN for '{w}':",neighbors(ft,w))


== Lexical coverage ==
labeled-sentiment_2col.xlsx: W2V=0.932, FT=0.932
test__1__2col.xlsx: W2V=0.987, FT=0.987
train__3__2col.xlsx: W2V=0.990, FT=0.990
train-00000-of-00001_2col.xlsx: W2V=0.943, FT=0.943
merged_dataset_CSV__1__2col.xlsx: W2V=0.949, FT=0.949

== Similarities ==
Synonyms: W2V=0.361, FT=0.424
Antonyms: W2V=0.310, FT=0.438

== Nearest neighbors ==
  W2V NN for 'yaxşı': ['iyi', '<RATING_POS>', 'yaxshi', 'awsome', 'yaxwi']
  FT  NN for 'yaxşı': ['yaxşıı', 'yaxşıkı', 'yaxşıca', 'yaxş', 'yaxşıya']
  W2V NN for 'pis': ['vərdişlərə', 'günd', 'yaxşıdır_NEG', 'bugunki', 'kardeşi']
  FT  NN for 'pis': ['piis', 'pi', 'pisdii', 'pixlr', 'pisleşdi']
  W2V NN for 'çox': ['çoox', 'çöx', 'əladir', 'gözəldir', 'çoxx']
  FT  NN for 'çox': ['çoxçox', 'çoxx', 'çoxh', 'ço', 'çoh']
  W2V NN for 'bahalı': ['metallarla', 'portretlerinə', 'yaxtaları', 'qabardılır', 'avropalaşırıq']
  FT  NN for 'bahalı': ['bahalıı', 'bahalısı', 'bahalıq', 'baharlı', 'bahalığı']
  W2V NN for 'ucuz': ['düzəltdiril